In [1]:
from pathlib import Path
from datasets import load_dataset

hf_cache_dir = Path(".hf_cache")
hf_cache_dir.mkdir(parents=True, exist_ok=True)
dataset = load_dataset("ts-arena/exchange_rate", cache_dir=str(hf_cache_dir))
print(dataset)


/Users/akarn/Documents/Exchange_curr_pred/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 1518/1518 [00:00<00:00, 761233.08 examples/s]

DatasetDict({
    train: Dataset({
        features: ['0', '1', '2', '3', '4', '5', '6', 'OT', 'timestamp_idx'],
        num_rows: 5311
    })
    validation: Dataset({
        features: ['0', '1', '2', '3', '4', '5', '6', 'OT', 'timestamp_idx'],
        num_rows: 759
    })
    test: Dataset({
        features: ['0', '1', '2', '3', '4', '5', '6', 'OT', 'timestamp_idx'],
        num_rows: 1518
    })
})


In [2]:
df_train=dataset["train"].to_pandas()
df_test=dataset["test"].to_pandas()
df_validation=dataset["validation"].to_pandas()

In [13]:
print(df_train.shape)
print(df_test.shape)
print(df_validation.shape)

(5311, 9)
(1518, 9)
(759, 9)


In [14]:
print(df_train.head())

          0         1         2         3        4         5         6  \
0  0.606785 -0.361671  0.735367 -1.164374  2.85189 -1.861369 -1.820047   
1  0.570900 -0.367639  0.729630 -1.170907  2.85189 -1.838665 -1.847258   
2  0.618423 -0.252456  0.728914 -1.027468  2.85189 -1.736953 -1.805130   
3  0.611634 -0.206502  0.738951 -1.007285  2.85189 -1.756932 -1.849738   
4  0.600966 -0.111013  0.738235 -0.953372  2.85189 -1.768738 -1.785181   

         OT  timestamp_idx  
0 -0.124081              0  
1 -0.113588              1  
2 -0.078960              2  
3 -0.082108              3  
4 -0.066368              4  


In [5]:
print(df_train.isnull().sum())

0                0
1                0
2                0
3                0
4                0
5                0
6                0
OT               0
timestamp_idx    0
dtype: int64


In [6]:
print(df_train.describe())

                  0             1             2            3             4  \
count  5.311000e+03  5.311000e+03  5.311000e+03  5311.000000  5.311000e+03   
mean  -2.873054e-08  1.149222e-08 -1.149222e-08     0.000000 -3.447665e-08   
std    1.000094e+00  1.000094e+00  1.000094e+00     1.000094  1.000094e+00   
min   -2.324163e+00 -1.786166e+00 -1.612920e+00    -1.983000 -1.047729e+00   
25%   -6.525500e-01 -7.650247e-01 -7.928607e-01    -0.789243 -6.088446e-01   
50%    1.902298e-01 -2.187425e-01 -2.229339e-01     0.037868 -6.066260e-01   
75%    5.495632e-01  7.696171e-01  7.558249e-01     0.673665  3.706311e-01   
max    2.473281e+00  2.610416e+00  2.955284e+00     2.469653  2.852769e+00   

                 5             6            OT  timestamp_idx  
count  5311.000000  5.311000e+03  5.311000e+03    5311.000000  
mean      0.000000 -5.746108e-09 -1.149222e-08    2655.000000  
std       1.000094  1.000094e+00  1.000094e+00    1533.297971  
min      -2.391725 -1.849738e+00 -2.22112

In [ ]:
print(dataset["train"].info)


DatasetInfo(description='', citation='', homepage='', license='', features={'0': Value('float32'), '1': Value('float32'), '2': Value('float32'), '3': Value('float32'), '4': Value('float32'), '5': Value('float32'), '6': Value('float32'), 'OT': Value('float32'), 'timestamp_idx': Value('int64')}, post_processed=None, supervised_keys=None, builder_name='parquet', dataset_name='exchange_rate', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=218416, num_examples=5311, shard_lengths=None, original_shard_lengths=None, dataset_name='exchange_rate'), 'validation': SplitInfo(name='validation', num_bytes=31215, num_examples=759, shard_lengths=None, original_shard_lengths=None, dataset_name='exchange_rate'), 'test': SplitInfo(name='test', num_bytes=62430, num_examples=1518, shard_lengths=None, original_shard_lengths=None, dataset_name='exchange_rate')}, download_checksums={'hf://datasets/ts-arena/exchange_rate@e731241b067e752a8a72100178741c441a018688/train.p

In [ ]:
# Division-by-zero diagnostics and guards
def safe_div(numerator, denominator, context):
    if denominator == 0:
        raise ValueError(f"Division by zero in {context}: denominator is 0")
    return numerator / denominator

print('train/val/test row counts:', len(df_train), len(df_validation), len(df_test))

for name in ['batch_size', 'train_batch_size', 'val_batch_size', 'test_batch_size']:
    if name in globals():
        val = globals()[name]
        print(f'{name}={val}')
        if isinstance(val, (int, float)) and val == 0:
            raise ValueError(f"{name} is 0. Set it to a positive value.")

for name in ['train_loader', 'val_loader', 'test_loader']:
    if name in globals():
        l = len(globals()[name])
        print(f'len({name})={l}')
        if l == 0:
            raise ValueError(
                f"{name} has 0 batches (likely batch_size too large with drop_last=True, or empty dataset)."
            )

print('Diagnostics complete: no zero denominators detected in common loader settings.')
